# Price Prediction Dataset EDA

This notebook visualizes correlations between the actual boundary variables and the standardized price target.

In [4]:
from pathlib import Path
import os


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        dataset_path = candidate / "output/clean_datasets/price_prediction_dataset.csv"
        if dataset_path.exists():
            return candidate
    raise FileNotFoundError("Could not find output/clean_datasets/price_prediction_dataset.csv")


PROJECT_ROOT = find_project_root()
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".matplotlib-cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import pandas as pd
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from IPython.display import display

INPUT_PATH = PROJECT_ROOT / "output/clean_datasets/price_prediction_dataset.csv"
OUTPUT_DIR = PROJECT_ROOT / "output/EDA_figures/price_prediction"
METHOD = "pearson"

plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 200
plt.rcParams["font.size"] = 10

print(f"Project root: {PROJECT_ROOT}")
print(f"Input dataset: {INPUT_PATH}")

Project root: /Users/lenovo/Desktop/GRIPS2026_Project
Input dataset: /Users/lenovo/Desktop/GRIPS2026_Project/output/clean_datasets/price_prediction_dataset.csv


## Load Dataset

In [5]:

df = pd.read_csv(INPUT_PATH, parse_dates=["time"])
numeric = df.select_dtypes(include="number")

print(f"Rows: {len(df):,}")
print(f"Numeric columns: {len(numeric.columns)}")
display(df.head())

Rows: 34,944
Numeric columns: 8


,time,System_Load,Wind_And_Solar,Tie_Line,Wind_Power,Photovoltaic,Hydropower,Non-Marketized_Unit,price
0,2025-01-02 00:00:00,2.972606,0.351390,0.051881,0.351626,-0.000236,0.026298,0.440875,1.272545
1,2025-01-02 00:15:00,2.975888,0.360655,0.039225,0.361107,-0.000452,0.026249,0.440386,1.272545
2,2025-01-02 00:30:00,2.969591,0.378003,0.041159,0.378530,-0.000527,0.026497,0.439392,1.272545
3,2025-01-02 00:45:00,2.976092,0.388225,0.028153,0.388598,-0.000373,0.026342,0.435306,1.272545
4,2025-01-02 01:00:00,2.959156,0.396585,0.042151,0.396982,-0.000396,0.026375,0.435755,1.272545


## Correlation Heatmap

In [6]:
corr = numeric.corr(method=METHOD)


def plot_correlation_heatmap(corr: pd.DataFrame, method: str = "pearson"):
    labels = corr.columns.tolist()
    fig, ax = plt.subplots(figsize=(10, 8))
    norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
    image = ax.imshow(corr, cmap="coolwarm", norm=norm)

    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    ax.set_title(f"{method.title()} Correlation Heatmap")

    for row_idx, row_label in enumerate(labels):
        for col_idx, col_label in enumerate(labels):
            value = corr.loc[row_label, col_label]
            text_color = "white" if abs(value) > 0.65 else "black"
            ax.text(
                col_idx,
                row_idx,
                f"{value:.2f}",
                ha="center",
                va="center",
                color=text_color,
                fontsize=9,
            )

    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label="Correlation")
    fig.tight_layout()
    return fig, ax


heatmap_fig, heatmap_ax = plot_correlation_heatmap(corr, METHOD)
display(heatmap_fig)

<Figure size 1300x1040 with 2 Axes>

## Correlation With Price

In [7]:
price_corr = corr["price"].drop("price").sort_values(key=lambda values: values.abs(), ascending=True)
colors = ["#b2182b" if value >= 0 else "#2166ac" for value in price_corr]

bar_fig, ax = plt.subplots(figsize=(8, 4.8))
ax.barh(price_corr.index, price_corr.values, color=colors)
ax.axvline(0, color="#333333", linewidth=1)
ax.set_xlim(-1, 1)
ax.set_xlabel("Correlation with price")
ax.set_title(f"{METHOD.title()} Correlation With Price")

for y_pos, value in enumerate(price_corr.values):
    label_x = value + 0.03 if value >= 0 else value - 0.03
    ha = "left" if value >= 0 else "right"
    ax.text(label_x, y_pos, f"{value:.2f}", va="center", ha=ha)

bar_fig.tight_layout()
display(bar_fig)

<Figure size 1040x624 with 1 Axes>

## Save Figures

In [8]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

matrix_path = OUTPUT_DIR / "EDA_price_prediction_correlation_matrix.csv"
heatmap_path = OUTPUT_DIR / "EDA_price_prediction_heatmap.png"
price_corr_path = OUTPUT_DIR / "EDA_price_prediction_price_correlations.png"

corr.to_csv(matrix_path)
heatmap_fig.savefig(heatmap_path, bbox_inches="tight")
bar_fig.savefig(price_corr_path, bbox_inches="tight")

print(f"Saved correlation matrix: {matrix_path}")
print(f"Saved heatmap: {heatmap_path}")
print(f"Saved price-correlation chart: {price_corr_path}")

Saved correlation matrix: /Users/lenovo/Desktop/GRIPS2026_Project/output/EDA_figures/price_prediction/EDA_price_prediction_correlation_matrix.csv
Saved heatmap: /Users/lenovo/Desktop/GRIPS2026_Project/output/EDA_figures/price_prediction/EDA_price_prediction_heatmap.png
Saved price-correlation chart: /Users/lenovo/Desktop/GRIPS2026_Project/output/EDA_figures/price_prediction/EDA_price_prediction_price_correlations.png


## Price Overview Visualization

This figure focuses directly on the `price` column from `price_prediction_dataset.csv`. It shows the daily price trend, the overall price distribution, and the typical intraday pattern by hour.

In [9]:
price_df = df[["time", "price"]].copy()
price_df = price_df.sort_values("time").dropna(subset=["time", "price"])
price_df["hour"] = price_df["time"].dt.hour

price_daily = (
    price_df.set_index("time")["price"]
    .resample("D")
    .agg(mean="mean", median="median", min="min", max="max")
    .dropna(how="all")
)

price_by_hour = (
    price_df.groupby("hour")["price"]
    .agg(
        q25=lambda values: values.quantile(0.25),
        median="median",
        q75=lambda values: values.quantile(0.75),
    )
    .reset_index()
)

display(price_df["price"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_frame("price"))

price_fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Daily trend: mean line with min-max band, so the yearly movement is readable.
axes[0, 0].plot(price_daily.index, price_daily["mean"], color="#1f77b4", linewidth=1.6, label="Daily mean")
axes[0, 0].fill_between(
    price_daily.index,
    price_daily["min"].to_numpy(),
    price_daily["max"].to_numpy(),
    color="#1f77b4",
    alpha=0.15,
    linewidth=0,
    label="Daily min-max",
)
axes[0, 0].set_title("Daily Price Trend")
axes[0, 0].set_xlabel("Time")
axes[0, 0].set_ylabel("Price")
axes[0, 0].grid(True, alpha=0.25)
axes[0, 0].legend()

# Overall distribution of all 15-minute price rows.
axes[0, 1].hist(price_df["price"], bins=50, color="#4c78a8", edgecolor="white", alpha=0.9)
axes[0, 1].axvline(price_df["price"].median(), color="#f58518", linestyle="--", linewidth=1.8, label="Median")
axes[0, 1].set_title("Price Distribution")
axes[0, 1].set_xlabel("Price")
axes[0, 1].set_ylabel("Count")
axes[0, 1].grid(True, alpha=0.25)
axes[0, 1].legend()

# Intraday pattern: hourly median with IQR band.
axes[1, 0].plot(price_by_hour["hour"], price_by_hour["median"], color="#54a24b", marker="o", label="Hourly median")
axes[1, 0].fill_between(
    price_by_hour["hour"],
    price_by_hour["q25"].to_numpy(),
    price_by_hour["q75"].to_numpy(),
    color="#54a24b",
    alpha=0.18,
    linewidth=0,
    label="Hourly IQR",
)
axes[1, 0].set_xticks(range(0, 24, 2))
axes[1, 0].set_title("Intraday Price Pattern")
axes[1, 0].set_xlabel("Hour of Day")
axes[1, 0].set_ylabel("Price")
axes[1, 0].grid(True, alpha=0.25)
axes[1, 0].legend()

# Raw 15-minute observations, lightly drawn to show density without over-emphasizing every point.
axes[1, 1].plot(price_df["time"], price_df["price"], color="#9467bd", linewidth=0.35, alpha=0.45)
axes[1, 1].set_title("Raw 15-Minute Price Series")
axes[1, 1].set_xlabel("Time")
axes[1, 1].set_ylabel("Price")
axes[1, 1].grid(True, alpha=0.25)

price_fig.suptitle("Price Overview from price_prediction_dataset.csv", fontsize=14, y=1.02)
price_fig.autofmt_xdate()
price_fig.tight_layout()

price_overview_path = OUTPUT_DIR / "EDA_price_prediction_price_overview.png"
price_fig.savefig(price_overview_path, bbox_inches="tight")
display(price_fig)

print(f"Saved price overview figure: {price_overview_path}")

,price
count,34944.000000
mean,0.940546
std,1.074452
min,-0.318136
1%,-0.159068
5%,-0.159068
25%,0.003181
50%,0.890781
75%,1.177104
95%,3.184543


<Figure size 1820x1040 with 4 Axes>

Saved price overview figure: /Users/lenovo/Desktop/GRIPS2026_Project/output/EDA_figures/price_prediction/EDA_price_prediction_price_overview.png
